# Agent 2 Dispatch Experiment: Directory vs. Online Search

## Research Question

Can Gemini select the correct marine wildlife response center from online information as reliably as it can when grounded in the official rescue-center directory?

This notebook evaluates **dispatch center choice only**. It does not replace the operational dispatch design. For a safety-critical system, the maintained directory remains the primary source and a human coordinator remains responsible for final dispatch approval.


## Evaluation Setup

This experiment uses **100 synthetic dispatch scenarios** grounded in the response areas and capabilities listed in the `2026 West Coast Marine Mammal Stranding Network Directory`.

| Coverage | Count |
|---|---:|
| Total dispatch-choice cases | 100 |
| Directory organizations represented | 33 |
| Live animal cases | 57 |
| Dead animal cases | 43 |

The benchmark includes pinnipeds, cetaceans, and sea turtles, as well as live rescue and dead-animal response needs. The featured Santa Barbara case is included as `D001`.

### Two Compared Conditions

1. **Directory-grounded selection:** Gemini receives the incident and the supplied rescue-center directory, then selects only from directory rows.
2. **Online-grounded selection:** Gemini receives the same incident but not the directory. It uses Google Search grounding to identify a responder and hotline from public web information.

The benchmark's directory row is the reference answer. Online search is evaluated as a possible **verification layer**, not as authority to dispatch.


In [2]:
from pathlib import Path
import json
import os
import re
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display, Markdown

PROJECT_DIR = Path.cwd()
SHARED_DIR = PROJECT_DIR.parent  # Coordination_agent2/ holds the shared directory and prompt library
BENCHMARK_PATH = PROJECT_DIR / "agent2_dispatch_100_case_benchmark.csv"
DIRECTORY_PATH = SHARED_DIR / "agent2_rescue_center_directory.csv"
PROMPT_PATH = SHARED_DIR / "agent2_prompt_library.json"

missing = [p.name for p in [BENCHMARK_PATH, DIRECTORY_PATH, PROMPT_PATH] if not p.exists()]
assert not missing, f"Upload the required files into this JupyterLab folder: {missing}"

benchmark = pd.read_csv(BENCHMARK_PATH).fillna("")
directory = pd.read_csv(DIRECTORY_PATH).fillna("")
prompt_library = json.loads(PROMPT_PATH.read_text(encoding="utf-8"))
prompts = {item["id"]: item["template"] for item in prompt_library["prompts"]}

assert len(benchmark) == 100, "The benchmark must contain 100 cases."
print("Benchmark cases:", len(benchmark))
print("Directory organizations represented:", benchmark["expected_center_name"].nunique())
display(benchmark["alive_status"].value_counts().rename_axis("animal_status").reset_index(name="cases"))
display(pd.crosstab(benchmark["alive_status"], benchmark["species_group"]))


Benchmark cases: 100
Directory organizations represented: 33


,animal_status,cases
0,live,57
1,dead,43


species_group,cetacean,pinniped,sea_turtle
alive_status,,,
dead,9,26,8
live,24,28,5


## Preview The Benchmark Cases

Each scenario is a dispatch-choice test case built from an official directory row. The `expected_center_name` and `expected_hotline` columns are the answer key used for evaluation only; they are never included in the online-search prompt.


In [3]:
display(
    benchmark[[
        "test_id", "place_name", "species_group", "alive_status",
        "reported_condition", "response_need", "expected_center_name", "expected_hotline"
    ]].head(12)
)


,test_id,place_name,species_group,alive_status,reported_condition,response_need,expected_center_name,expected_hotline
0,D001,"Santa Barbara Beach, CA",pinniped,live,live California sea lion with a visible wound ...,live pinniped rescue and rehabilitation,Channel Islands Marine & Wildlife Institute (C...,805-567-1505
1,D002,"Del Norte and Humboldt Counties, California",pinniped,live,live injured sea lion with a visible wound,live pinniped rescue,Northcoast Marine Mammal Center (NMMC),707-951-4722
2,D003,"Del Norte and Humboldt Counties, California",cetacean,live,live stranded dolphin showing limited mobility,live cetacean response,Northcoast Marine Mammal Center (NMMC),707-951-4722
3,D004,"Del Norte and Humboldt Counties, California",sea_turtle,live,live stranded sea turtle needing assessment,live sea turtle response,Northcoast Marine Mammal Center (NMMC),707-951-4722
4,D005,"Del Norte, Humboldt, and Northern Mendocino Co...",pinniped,dead,dead stranded seal on the beach,dead pinniped response,California Polytechnic University - Humboldt (...,707-826-3650 or 707-498-6200
5,D006,"Del Norte, Humboldt, and Northern Mendocino Co...",cetacean,dead,dead dolphin on the shoreline,dead cetacean response,California Polytechnic University - Humboldt (...,707-826-3650 or 707-498-6200
6,D007,"Del Norte, Humboldt, and Northern Mendocino Co...",sea_turtle,dead,dead sea turtle on shoreline,dead sea turtle response,California Polytechnic University - Humboldt (...,707-826-3650 or 707-498-6200
7,D008,"San Francisco Bay: Mendocino, Sonoma, Napa, Ma...",pinniped,live,live injured sea lion with a visible wound,live pinniped rescue,The Marine Mammal Center - Northern Range Oper...,415-289-7350
8,D009,"San Francisco Bay: Mendocino, Sonoma, Napa, Ma...",sea_turtle,live,live stranded sea turtle needing assessment,live sea turtle response,The Marine Mammal Center - Northern Range Oper...,415-289-7350
9,D010,"San Francisco Bay: Mendocino, Sonoma, Napa, Ma...",cetacean,live,live whale entangled in fishing gear,whale entanglement response,The Marine Mammal Center - Northern Range Oper...,415-289-7350


## Configure Vertex AI Gemini

Start with only a small batch to confirm access and output format. The full experiment runs two model calls per case, for a total of up to **200 Gemini calls**, and Google Search grounding may add usage cost.


In [ ]:
GOOGLE_CLOUD_PROJECT = os.getenv("GOOGLE_CLOUD_PROJECT", "spring-2026-lifejacket")
GOOGLE_CLOUD_LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION", "global")
VERTEX_GEMINI_MODEL = os.getenv("VERTEX_GEMINI_MODEL", "publishers/google/models/gemini-3.5-flash")
SEARCH_MODEL = os.getenv("VERTEX_SEARCH_MODEL", VERTEX_GEMINI_MODEL)
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

# Recommended sequence: run 1 featured case, then 5 cases, then all 100 cases.
TEST_LIMIT = 100
RUN_DIRECTORY_EXPERIMENT = False
RUN_ONLINE_EXPERIMENT = False

print("Project:", GOOGLE_CLOUD_PROJECT)
print("Directory-selection model:", VERTEX_GEMINI_MODEL)
print("Online-search model:", SEARCH_MODEL)
print("Current case limit:", TEST_LIMIT)


## Helpers And Output Contract

Both methods return the same dispatch-choice fields. Scoring is deliberately strict for center identity and hotline because a wrong contact number is a safety-relevant error.


In [5]:
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project=GOOGLE_CLOUD_PROJECT, location=GOOGLE_CLOUD_LOCATION)

DISPATCH_SCHEMA = {
    "type": "object",
    "properties": {
        "dispatch_center_name": {"type": "string"},
        "dispatch_hotline": {"type": "string"},
        "dispatch_response_area": {"type": "string"},
        "dispatch_response_type": {"type": "string"},
        "dispatch_match_confidence": {"type": "number", "minimum": 0, "maximum": 1},
        "dispatch_match_reason": {"type": "string"},
        "source_urls": {"type": "array", "items": {"type": "string"}},
    },
    "required": [
        "dispatch_center_name", "dispatch_hotline", "dispatch_response_area",
        "dispatch_response_type", "dispatch_match_confidence",
        "dispatch_match_reason", "source_urls"
    ],
}

USAGE_LOG = []

def as_json(value):
    return json.dumps(value, indent=2, ensure_ascii=False, default=str)

def normalize_text(value):
    return re.sub(r"[^a-z0-9]+", " ", str(value).lower()).strip()

def normalize_phone(value):
    return re.sub(r"[^0-9]", "", str(value))

def acronym(value):
    match = re.search(r"\(([^)]+)\)", str(value))
    return normalize_text(match.group(1)) if match else ""

def center_agrees(predicted, expected):
    predicted_norm, expected_norm = normalize_text(predicted), normalize_text(expected)
    expected_acronym = acronym(expected)
    return (
        predicted_norm == expected_norm
        or expected_norm in predicted_norm
        or predicted_norm in expected_norm
        or (expected_acronym and expected_acronym in predicted_norm)
    )

def usage_values(response):
    usage = getattr(response, "usage_metadata", None)
    return {
        "input_tokens": getattr(usage, "prompt_token_count", None) if usage else None,
        "output_tokens": getattr(usage, "candidates_token_count", None) if usage else None,
        "reasoning_tokens": getattr(usage, "thoughts_token_count", None) if usage else None,
    }

def parse_json_text(text):
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("Response did not contain a JSON object.")
    return json.loads(text[start:end + 1])

def case_packet(row):
    return {
        "test_id": row["test_id"],
        "place_name": row["place_name"],
        "species_group": row["species_group"],
        "animal_status": row["alive_status"],
        "reported_condition": row["reported_condition"],
        "response_need": row["response_need"],
    }

def score_row(case, output, method, response):
    center_correct = center_agrees(output["dispatch_center_name"], case["expected_center_name"])
    hotline_correct = normalize_phone(output["dispatch_hotline"]) == normalize_phone(case["expected_hotline"])
    return {
        "test_id": case["test_id"],
        "method": method,
        "species_group": case["species_group"],
        "alive_status": case["alive_status"],
        "expected_center": case["expected_center_name"],
        "predicted_center": output["dispatch_center_name"],
        "expected_hotline": case["expected_hotline"],
        "predicted_hotline": output["dispatch_hotline"],
        "center_correct": center_correct,
        "hotline_correct": hotline_correct,
        "full_dispatch_correct": center_correct and hotline_correct,
        "reason": output["dispatch_match_reason"],
        "source_urls": "; ".join(output.get("source_urls", [])),
        **usage_values(response),
    }

def metric_summary(results):
    return pd.DataFrame([
        {"metric": "Center match rate", "correct": int(results["center_correct"].sum()), "total": len(results), "accuracy": results["center_correct"].mean()},
        {"metric": "Hotline match rate", "correct": int(results["hotline_correct"].sum()), "total": len(results), "accuracy": results["hotline_correct"].mean()},
        {"metric": "Full dispatch match rate", "correct": int(results["full_dispatch_correct"].sum()), "total": len(results), "accuracy": results["full_dispatch_correct"].mean()},
    ])


## Condition A: Dispatch Using The Official Directory

This is the operationally grounded method. Gemini receives the authoritative directory rows and is instructed to choose only from those rows.


In [ ]:
def run_directory_condition(cases):
    results = []
    directory_payload = directory.to_dict(orient="records")
    for _, case in cases.iterrows():
        prompt = f'''You are the LifeJacket Dispatch Center Selection Module.

Choose the best responder for this incident using ONLY the provided official directory rows.
Do not invent an organization or hotline. Match geographic coverage, species group,
animal status, required response capability, and contact availability.

Return JSON only with keys:
- dispatch_center_name
- dispatch_hotline
- dispatch_response_area
- dispatch_response_type
- dispatch_match_confidence
- dispatch_match_reason
- source_urls (return an empty list for this directory condition)

Incident:
{as_json(case_packet(case))}

Official directory rows:
{as_json(directory_payload)}
'''
        response = client.models.generate_content(
            model=VERTEX_GEMINI_MODEL,
            contents=prompt,
            config={
                "temperature": 0.1,
                "response_mime_type": "application/json",
                "response_json_schema": DISPATCH_SCHEMA,
            },
        )
        output = json.loads(response.text)
        results.append(score_row(case, output, "official_directory", response))
    return pd.DataFrame(results)

cases_to_run = benchmark.head(TEST_LIMIT) if TEST_LIMIT else benchmark
if RUN_DIRECTORY_EXPERIMENT:
    directory_results = run_directory_condition(cases_to_run)
    display(directory_results)
    display(metric_summary(directory_results))
else:
    display(Markdown("Set `RUN_DIRECTORY_EXPERIMENT = True` to run the directory-grounded condition."))


## Condition B: Dispatch Using Online Search

This condition intentionally withholds the official directory from Gemini. Google Search grounding is enabled, and the model is told to prioritize official responder or government sources. The generated result is compared back to the directory answer key only after it is returned.


In [ ]:
def grounding_urls(response):
    urls = []
    metadata = getattr(response.candidates[0], "grounding_metadata", None)
    chunks = getattr(metadata, "grounding_chunks", None) or []
    for chunk in chunks:
        web = getattr(chunk, "web", None)
        uri = getattr(web, "uri", None)
        if uri:
            urls.append(uri)
    return urls

def run_online_condition(cases):
    results = []
    for _, case in cases.iterrows():
        prompt = f'''You are evaluating online dispatch lookup for a marine wildlife rescue system.

Search current public information to identify the organization responsible for this incident.
Prioritize official responder websites and NOAA or state agency sources.
Do not assume a result without source evidence. Do not provide care instructions.

Return one JSON object only with keys:
- dispatch_center_name
- dispatch_hotline
- dispatch_response_area
- dispatch_response_type
- dispatch_match_confidence
- dispatch_match_reason
- source_urls

Incident:
{as_json(case_packet(case))}
'''
        response = client.models.generate_content(
            model=SEARCH_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.2,
                tools=[types.Tool(google_search=types.GoogleSearch())],
            ),
        )
        output = parse_json_text(response.text)
        output["source_urls"] = list(dict.fromkeys(output.get("source_urls", []) + grounding_urls(response)))
        results.append(score_row(case, output, "online_search", response))
    return pd.DataFrame(results)

if RUN_ONLINE_EXPERIMENT:
    online_results = run_online_condition(cases_to_run)
    display(online_results)
    display(metric_summary(online_results))
else:
    display(Markdown("Set `RUN_ONLINE_EXPERIMENT = True` to run the Google Search-grounded condition."))


## Recommended Run Sequence

Run this evaluation in stages so errors or cost remain manageable:

1. Set `TEST_LIMIT = 1` and enable both conditions to test the featured Santa Barbara case.
2. Set `TEST_LIMIT = 5` to verify format and scoring on several cases.
3. Set `TEST_LIMIT = None` only when ready for the complete 100-case comparison.

The complete experiment may produce up to **200 model calls**: 100 directory-grounded dispatch choices and 100 online-grounded dispatch choices.


In [11]:
#Example switches for a one-case trial:
TEST_LIMIT = 100
RUN_DIRECTORY_EXPERIMENT = True
RUN_ONLINE_EXPERIMENT = True
#
# After changing the variables above, rerun the two condition cells.


## Compare Results

Exact matching can be deliberately strict in areas where directory jurisdictions overlap. A mismatch should be reviewed to distinguish an unsafe error from a different but potentially compatible authorized responder.


In [12]:
if "directory_results" in globals() and "online_results" in globals():
    combined = pd.concat([directory_results, online_results], ignore_index=True)
    comparison = combined.groupby("method", as_index=False).agg(
        tested_cases=("test_id", "count"),
        center_match_rate=("center_correct", "mean"),
        hotline_match_rate=("hotline_correct", "mean"),
        full_dispatch_match_rate=("full_dispatch_correct", "mean"),
    )
    display(comparison)

    paired = directory_results[["test_id", "full_dispatch_correct"]].merge(
        online_results[["test_id", "full_dispatch_correct"]],
        on="test_id",
        suffixes=("_directory", "_online"),
    )
    display(paired.value_counts(["full_dispatch_correct_directory", "full_dispatch_correct_online"]).reset_index(name="cases"))

    online_mismatches = online_results.loc[~online_results["full_dispatch_correct"]]
    display(Markdown("### Online Search Mismatches Requiring Review"))
    display(online_mismatches[[
        "test_id", "expected_center", "predicted_center",
        "expected_hotline", "predicted_hotline", "source_urls", "reason"
    ]])
else:
    display(Markdown("Run both experimental conditions before calculating the comparison."))


Run both experimental conditions before calculating the comparison.

## Compare Live Versus Dead-Animal Performance

This breakdown tests whether online search performs differently when the correct responder is responsible for dead-animal response rather than live rescue.


In [13]:
if "directory_results" in globals() and "online_results" in globals():
    status_comparison = pd.concat([directory_results, online_results], ignore_index=True).groupby(
        ["method", "alive_status"], as_index=False
    ).agg(
        cases=("test_id", "count"),
        center_match_rate=("center_correct", "mean"),
        hotline_match_rate=("hotline_correct", "mean"),
        full_dispatch_match_rate=("full_dispatch_correct", "mean"),
    )
    display(status_comparison)


## Token Usage And Approximate Cost

Token usage is collected from model response metadata when available. Online grounding can include additional provider charges; consult the current Vertex AI pricing page before presenting a final cost number.


In [14]:
INPUT_PRICE_PER_MILLION = 1.50
OUTPUT_PRICE_PER_MILLION = 9.00

if "directory_results" in globals() and "online_results" in globals():
    usage = pd.concat([directory_results, online_results], ignore_index=True)
    usage["estimated_token_cost_usd"] = (
        usage["input_tokens"].fillna(0) * INPUT_PRICE_PER_MILLION / 1_000_000
        + (usage["output_tokens"].fillna(0) + usage["reasoning_tokens"].fillna(0)) * OUTPUT_PRICE_PER_MILLION / 1_000_000
    )
    display(usage.groupby("method", as_index=False)[
        ["input_tokens", "output_tokens", "reasoning_tokens", "estimated_token_cost_usd"]
    ].sum())
    print("This estimate excludes any separately billed online-grounding/search fee.")


## Export Results

Run after completing either or both conditions.


In [15]:
written = []
if "directory_results" in globals():
    directory_results.to_csv(PROJECT_DIR / "agent2_dispatch_directory_results.csv", index=False)
    written.append("agent2_dispatch_directory_results.csv")
if "online_results" in globals():
    online_results.to_csv(PROJECT_DIR / "agent2_dispatch_online_results.csv", index=False)
    written.append("agent2_dispatch_online_results.csv")
if "comparison" in globals():
    comparison.to_csv(PROJECT_DIR / "agent2_dispatch_method_comparison.csv", index=False)
    written.append("agent2_dispatch_method_comparison.csv")
print("Exported:", written or "No results have been run yet.")


Exported: No results have been run yet.


## How To Present This Experiment

> "I tested one specific design choice: whether Gemini should dispatch from an authoritative rescue-center directory or search online for a responder. I created 100 scenario cases from the official directory, including 43 dead-animal cases, and ran the same dispatch-selection task in two conditions. The directory-grounded condition gives Gemini the approved source; the online condition uses Google Search grounding without seeing the answer key. I compare organization and hotline matches, then manually review mismatches in overlapping jurisdictions. Even if online search performs well, my operational design retains the directory as the primary source and uses online evidence only for verification or update detection."
